# Day 30 - Final Lakehouse Pipeline

**Topic:** Final Lakehouse Pipeline  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** รวม pattern หลักของ challenge เป็น mini Lakehouse pipeline เดียว ตั้งแต่ Bronze ingestion, Silver cleansing, DQ/quarantine, incremental MERGE, Gold aggregation, audit log และ streaming extension note

วันนี้เป็นวันปิด challenge โดยเราจะต่อภาพจากหลายวันที่ผ่านมาให้กลายเป็น Lakehouse pipeline แบบ end-to-end ขนาดย่อม ที่รันได้ใน Microsoft Fabric Notebook + Fabric Lakehouse โดยเป้าหมายไม่ใช่การสร้าง production framework เต็มรูปแบบ แต่คือการฝึกคิด end-to-end ว่า source data หนึ่งชุดควรถูก ingest, clean, validate, merge, aggregate และตรวจสอบผลลัพธ์อย่างไร

## Goal of the day

1. รวม Bronze → Silver → Gold pattern ให้เป็น flow เดียวที่รันใน Fabric Lakehouse ได้
2. สร้าง audit columns และ run log เพื่อ trace ได้ว่า pipeline run นี้ทำอะไรไปบ้าง
3. แยก valid records กับ invalid/quarantine records ด้วย DQ rules แบบ record-level
4. ใช้ watermark + MERGE เพื่อจำลอง incremental upsert แบบ rerun-safe
5. ใช้ `.explain()` และ schema drift check เพื่อปิดท้าย mindset เรื่อง production debugging


## Why it matters in production

ใน production pipeline งาน Data Engineering ไม่ได้จบแค่ transform DataFrame ให้ได้ผลลัพธ์ถูกต้อง แต่ต้องตอบคำถามเหล่านี้ได้ด้วย:

- ข้อมูลมาจาก source ไหน และถูก load เข้ามาในรอบไหน
- record ไหน valid, record ไหน invalid และไม่ผ่าน rule อะไร
- ถ้า rerun pipeline แล้ว target table จะยังถูกต้องและไม่เกิด duplicate ไหม
- ถ้า source schema เปลี่ยน จะตรวจเจอก่อนกระทบ target table หรือ downstream logic ไหม
- Gold table ที่ business ใช้งาน สร้างมาจาก transformation logic ไหน
- ถ้า job ช้า จะเริ่มตรวจจาก execution plan, join strategy และ aggregation step ตรงไหนก่อน

Production takeaway:

> Final pipeline ที่ดีไม่ใช่แค่เขียนสำเร็จ แต่ต้องตรวจสอบได้, rerun ได้, debug ได้ และอธิบาย flow ได้ตั้งแต่ source ถึง Gold


## Key concepts

| Concept | Meaning |
|---|---|
| Bronze table | เก็บข้อมูลจาก source ในรูปแบบ raw as ingested พร้อม audit columns เพื่อ trace และ reprocess ได้ |
| Silver cleansing | ทำ type casting, standardization, deduplication และเตรียมข้อมูลให้พร้อม validate |
| Data quality checks | ตรวจ required fields, valid dates, positive amount, duplicate key และ referential integrity |
| Quarantine table | เก็บ records ที่ไม่พร้อมไป downstream พร้อม `dq_issue_array` และ trace columns |
| Watermark | ค่า timestamp ล่าสุดที่ pipeline process สำเร็จ ใช้เลือก incremental window |
| MERGE / UPSERT | เขียน target แบบ update ถ้ามี key เดิม และ insert ถ้าเป็น key ใหม่ |
| Gold aggregation | ตารางสรุปที่พร้อมใช้ต่อสำหรับ reporting / analytics |
| Audit run log | log ระดับ pipeline run เช่น row counts, status, watermark และ run id |
| Schema drift detection | ตรวจ source columns ที่ missing/extra ก่อนปล่อยให้ pipeline เขียนข้อมูลผิด schema |
| Execution plan | แผนที่ Spark ใช้อธิบายว่าจะ execute transformations อย่างไร เช่น join, aggregation, shuffle และ scan |


## Concept explanation

### Final Lakehouse Pipeline คืออะไร?

Final pipeline ใน lab นี้คือการเอา pattern หลักจาก challenge มาต่อกันเป็น flow เดียว:

1. รับ mock source data เข้าสู่ Bronze
2. เพิ่ม audit columns เช่น `batch_id`, `run_id`, `ingestion_timestamp`, `source_file`
3. Clean และ standardize เป็น Silver candidate
4. Apply DQ checks แล้ว split เป็น valid / quarantine
5. ใช้ watermark เลือกเฉพาะ changed records
6. MERGE valid incremental records เข้า current target table
7. สร้าง Gold aggregate สำหรับ reporting
8. เขียน audit/run log และ schema validation log

พูดง่าย ๆ:

> Bronze = trace ได้  
> Silver = clean และ validate ได้  
> Gold = business ใช้ต่อได้  
> Audit/Watermark = run ซ้ำและตรวจสอบย้อนหลังได้

### ทำไมต้องมีทั้ง DQ และ quarantine?

ถ้าเราใช้ `.filter()` เอาเฉพาะ record ดี ๆ แล้วทิ้ง record เสียไปเลย จะเกิด silent data loss ทันที ใน lab นี้เราจึงเขียน invalid records ลง quarantine table พร้อมเหตุผล เช่น:

- `INVALID_AMOUNT`
- `INVALID_TRANSACTION_DATE`
- `CUSTOMER_NOT_FOUND`
- `PRODUCT_NOT_FOUND`
- `PROPERTY_NOT_FOUND`
- `DUPLICATE_BUSINESS_KEY_SUPERSEDED`

### Incremental MERGE ใน lab นี้คิดยังไง?

เราจำลอง control table ที่เก็บ `last_successful_watermark` แล้วเลือกเฉพาะ records ที่อยู่ใน window นี้:

```text
last_successful_watermark < updated_at_ts <= run_upper_bound
```

จากนั้นใช้ `MERGE INTO` เพื่อ update/insert target table ตาม `transaction_id`

> Mindset สำคัญ: อย่า update watermark ก่อน target write / merge สำเร็จ

### Batch pipeline ต่อไป streaming ได้ยังไง?

Day 30 ยังเป็น batch lab แต่ pattern ส่วน Bronze สามารถต่อยอดเป็น Structured Streaming ได้ โดยเปลี่ยนจาก batch read เป็น `readStream` และเขียน Bronze ด้วย checkpoint path ที่ชัดเจน ส่วน Silver/DQ/Gold อาจยังทำแบบ batch ตามรอบ หรือค่อย ๆ ขยายเป็น streaming ทีละส่วน


## Mermaid diagram: Final Lakehouse Pipeline Flow

```mermaid
flowchart LR
    A[Mock Source Batch] --> M[Schema Drift Check]
    M --> B[Bronze Transactions Table]
    M --> N[Schema Validation Log]

    B --> C[Silver Cleansing and Standardization]
    C --> D[DQ Checks]

    I[Customer/Product/Property Reference Tables] --> D

    D --> E[Silver Valid Records]
    D --> F[Quarantine Table]
    D --> L[DQ Summary Table]

    J[Watermark Control Table] --> W[Apply Watermark Window for MERGE]
    E --> W
    W --> G[MERGE into Silver Current Table]
    G --> H[Gold Daily Revenue Summary]

    H --> K[Audit Run Log]
    H --> U[Update Watermark After Success]
    U --> J

    O[Future Streaming Source] -.readStream.-> B
    P[Checkpoint Path] -.progress and recovery.-> O
```

Key idea:

> Pipeline นี้ยังเป็น learning lab แบบ batch แต่การแยก Bronze, Silver, Gold พร้อม audit log, DQ summary, quarantine table และ watermark ทำให้เห็นโครงสร้าง end-to-end ที่สามารถต่อยอดเป็น streaming ingestion ได้ภายหลัง โดยต้องเพิ่ม streaming source, checkpoint path, trigger strategy และ monitoring สำหรับ micro-batch เพิ่มเข้ามา


## Setup / imports

In [1]:
from datetime import date, datetime
from decimal import Decimal
from uuid import uuid4

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 3, Finished, Available, Finished, False)

## Create mock data

Data ที่ใช้มี 4 กลุ่ม:

1. transactions source data
2. customer reference table
3. product reference table
4. property reference table

Transactions จะตั้งใจใส่ปัญหาหลายแบบ เช่น negative amount, bad date, missing customer, invalid reference และ duplicate transaction key เพื่อให้ pipeline ได้ฝึก DQ + quarantine จริง


In [3]:
# Lab table names
bronze_table = "day30_bronze_transactions"
silver_stage_table = "day30_silver_transactions_stage"
silver_current_table = "day30_silver_transactions_current"
quarantine_table = "day30_quarantine_transactions"
gold_table = "day30_gold_daily_revenue"
dq_summary_table = "day30_dq_summary"
audit_log_table = "day30_audit_run_log"
watermark_table = "day30_watermark_control"
schema_log_table = "day30_schema_validation_log"

customer_table = "day30_dim_customers"
product_table = "day30_dim_products"
property_table = "day30_dim_properties"

pipeline_name = "day30_final_lakehouse_pipeline"
batch_id = "batch_2026_06_14_001"
run_id = f"run_{uuid4().hex[:8]}"  # Use the first 8 chars of the UUID hex string

print("run_id:", run_id)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 5, Finished, Available, Finished, False)

run_id: run_714826e2


In [4]:
transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), True),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("product_id", T.StringType(), True),
    T.StructField("property_id", T.StringType(), True),
    T.StructField("transaction_date", T.StringType(), True),
    T.StructField("amount", T.StringType(), True),
    T.StructField("currency", T.StringType(), True),
    T.StructField("payment_status", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("source_file", T.StringType(), True),
])

transactions_data = [
    ("T001", "C001", "P01", "PR01", "2026-06-01", "1200.50", "THB", "SUCCESS", "crm", "2026-06-01 08:00:00", "transactions_batch_001.csv"),
    ("T002", "C002", "P02", "PR01", "2026-06-01", "-50.00", "THB", "SUCCESS", "pos", "2026-06-01 09:00:00", "transactions_batch_001.csv"),
    ("T003", "C003", "P03", "PR02", "2026-06-02", "850.00", "THB", "PAID", "web", "2026-06-02 10:00:00", "transactions_batch_001.csv"),
    ("T004", "C999", "P01", "PR01", "2026-06-02", "450.00", "THB", "SUCCESS", "crm", "2026-06-02 11:00:00", "transactions_batch_001.csv"),
    ("T005", "C001", "P99", "PR02", "2026-06-03", "300.00", "THB", "SUCCESS", "web", "2026-06-03 08:30:00", "transactions_batch_001.csv"),
    ("T006", "C004", "P02", "PR99", "2026-06-04", "1500.00", "THB", "SUCCESS", "crm", "2026-06-04 12:00:00", "transactions_batch_001.csv"),
    ("T007", None, "P01", "PR01", "2026-06-04", "200.00", "THB", "SUCCESS", "crm", "2026-06-04 13:00:00", "transactions_batch_001.csv"),
    ("T008", "C002", "P02", "PR01", "bad-date", "400.00", "THB", "SUCCESS", "crm", "2026-06-04 14:00:00", "transactions_batch_001.csv"),
    ("T009", "C001", "P01", "PR01", "2026-06-05", "500.00", "THB", "SUCCESS", "crm", "2026-06-05 10:00:00", "transactions_batch_001.csv"),
    ("T009", "C001", "P01", "PR01", "2026-06-05", "650.00", "THB", "SUCCESS", "crm", "2026-06-05 11:00:00", "transactions_batch_001.csv"),
    ("T010", "C003", "P03", "PR02", "2026-06-06", "900.00", "THB", "SETTLED", "web", "2026-06-06 09:00:00", "transactions_batch_001.csv"),
]

df_transactions_raw = spark.createDataFrame(transactions_data, transaction_schema)

df_transactions_raw.show(truncate=False)
df_transactions_raw.printSchema()
print("Raw transaction rows:", df_transactions_raw.count())

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 6, Finished, Available, Finished, False)

+--------------+-----------+----------+-----------+----------------+-------+--------+--------------+-------------+-------------------+--------------------------+
|transaction_id|customer_id|product_id|property_id|transaction_date|amount |currency|payment_status|source_system|updated_at         |source_file               |
+--------------+-----------+----------+-----------+----------------+-------+--------+--------------+-------------+-------------------+--------------------------+
|T001          |C001       |P01       |PR01       |2026-06-01      |1200.50|THB     |SUCCESS       |crm          |2026-06-01 08:00:00|transactions_batch_001.csv|
|T002          |C002       |P02       |PR01       |2026-06-01      |-50.00 |THB     |SUCCESS       |pos          |2026-06-01 09:00:00|transactions_batch_001.csv|
|T003          |C003       |P03       |PR02       |2026-06-02      |850.00 |THB     |PAID          |web          |2026-06-02 10:00:00|transactions_batch_001.csv|
|T004          |C999       |

In [5]:
customers_data = [
    ("C001", "Alice Co", "Bangkok", "active"),
    ("C002", "Bob Retail", "Chiang Mai", "active"),
    ("C003", "Chana Travel", "Phuket", "active"),
    ("C004", "Delta Foods", "Rayong", "active"),
]

products_data = [
    ("P01", "Motor Policy", "Motor"),
    ("P02", "Health Policy", "Health"),
    ("P03", "Travel Policy", "Travel"),
]

properties_data = [
    ("PR01", "Bangkok Branch", "Bangkok", "Branch"),
    ("PR02", "Phuket Branch", "Phuket", "Branch"),
]

df_customers = spark.createDataFrame(customers_data, ["customer_id", "customer_name", "customer_region", "customer_status"])
df_products = spark.createDataFrame(products_data, ["product_id", "product_name", "product_category"])
df_properties = spark.createDataFrame(properties_data, ["property_id", "property_name", "property_region", "property_type"])

# Save reference tables as Lakehouse tables for join/read pattern practice.
df_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(customer_table)
df_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(product_table)
df_properties.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(property_table)

spark.read.table(customer_table).show(truncate=False)
spark.read.table(product_table).show(truncate=False)
spark.read.table(property_table).show(truncate=False)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 7, Finished, Available, Finished, False)

+-----------+-------------+---------------+---------------+
|customer_id|customer_name|customer_region|customer_status|
+-----------+-------------+---------------+---------------+
|C002       |Bob Retail   |Chiang Mai     |active         |
|C003       |Chana Travel |Phuket         |active         |
|C004       |Delta Foods  |Rayong         |active         |
|C001       |Alice Co     |Bangkok        |active         |
+-----------+-------------+---------------+---------------+

+----------+-------------+----------------+
|product_id|product_name |product_category|
+----------+-------------+----------------+
|P02       |Health Policy|Health          |
|P03       |Travel Policy|Travel          |
|P01       |Motor Policy |Motor           |
+----------+-------------+----------------+

+-----------+--------------+---------------+-------------+
|property_id|property_name |property_region|property_type|
+-----------+--------------+---------------+-------------+
|PR01       |Bangkok Branch|Bangk

## PySpark code examples

ใน section นี้จะรัน pipeline ตั้งแต่ Bronze → Silver → DQ/quarantine → incremental MERGE → Gold → audit log ตามลำดับ

### Example 1: Bronze ingestion with audit columns

Bronze table ควรเก็บข้อมูลจาก source ให้ trace ได้ว่า record มาจาก batch/run ไหน

Audit columns ที่ใช้ใน lab นี้:

- `batch_id`
- `run_id`
- `ingestion_timestamp`
- `source_file`
- `source_system`


In [6]:
df_bronze = (
    df_transactions_raw
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("run_id", F.lit(run_id))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_system", F.upper(F.trim(F.col("source_system"))))
)

(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

df_bronze_read = spark.read.table(bronze_table)

df_bronze_read.show(truncate=False)
print("Bronze rows:", df_bronze_read.count())

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 8, Finished, Available, Finished, False)

+--------------+-----------+----------+-----------+----------------+-------+--------+--------------+-------------+-------------------+--------------------------+--------------------+------------+--------------------------+
|transaction_id|customer_id|product_id|property_id|transaction_date|amount |currency|payment_status|source_system|updated_at         |source_file               |batch_id            |run_id      |ingestion_timestamp       |
+--------------+-----------+----------+-----------+----------------+-------+--------+--------------+-------------+-------------------+--------------------------+--------------------+------------+--------------------------+
|T009          |C001       |P01       |PR01       |2026-06-05      |650.00 |THB     |SUCCESS       |CRM          |2026-06-05 11:00:00|transactions_batch_001.csv|batch_2026_06_14_001|run_714826e2|2026-06-20 11:02:44.001917|
|T010          |C003       |P03       |PR02       |2026-06-06      |900.00 |THB     |SETTLED       |WEB     

### Example 2: Silver cleansing and latest-record selection

Silver stage จะทำสิ่งหลัก ๆ ต่อไปนี้:

- trim / upper / lower fields ที่ควร standardize
- cast `amount` เป็น `decimal(12,2)`
- parse `transaction_date` และ `updated_at`
- ใช้ window + `row_number()` เพื่อเก็บ latest record ต่อ `transaction_id`
- แยก duplicate records ที่ถูก superseded (ถูกแทนที่ด้วย record ล่าสุดของ key เดียวกัน) ไป quarantine ภายหลัง


In [7]:
df_silver_base = (
    df_bronze_read
    .withColumn("transaction_id", F.upper(F.trim(F.col("transaction_id"))))
    .withColumn("customer_id", F.upper(F.trim(F.col("customer_id"))))
    .withColumn("product_id", F.upper(F.trim(F.col("product_id"))))
    .withColumn("property_id", F.upper(F.trim(F.col("property_id"))))
    .withColumn("currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("payment_status", F.lower(F.trim(F.col("payment_status"))))
    .withColumn("transaction_date", F.to_date(F.col("transaction_date"), "yyyy-MM-dd"))
    .withColumn("amount", F.col("amount").cast(T.DecimalType(12, 2)))
    .withColumn("updated_at_ts", F.to_timestamp(F.col("updated_at"), "yyyy-MM-dd HH:mm:ss"))
)

latest_window = Window.partitionBy("transaction_id").orderBy(
    F.col("updated_at_ts").desc_nulls_last(),
    F.col("ingestion_timestamp").desc_nulls_last()
)

df_ranked = df_silver_base.withColumn("record_rank", F.row_number().over(latest_window))

df_latest = df_ranked.filter(F.col("record_rank") == 1).drop("record_rank")
df_duplicate_superseded = (
    df_ranked
    .filter(F.col("record_rank") > 1)
    .drop("record_rank")
    .withColumn("dq_issue_array", F.array(F.lit("DUPLICATE_BUSINESS_KEY_SUPERSEDED")))
    .withColumn("record_status", F.lit("quarantine"))
)

print("Latest candidate rows:", df_latest.count())
print("Superseded duplicate rows:", df_duplicate_superseded.count())

df_latest.orderBy("transaction_id").show(truncate=False)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 9, Finished, Available, Finished, False)

Latest candidate rows: 10
Superseded duplicate rows: 1
+--------------+-----------+----------+-----------+----------------+-------+--------+--------------+-------------+-------------------+--------------------------+--------------------+------------+--------------------------+-------------------+
|transaction_id|customer_id|product_id|property_id|transaction_date|amount |currency|payment_status|source_system|updated_at         |source_file               |batch_id            |run_id      |ingestion_timestamp       |updated_at_ts      |
+--------------+-----------+----------+-----------+----------------+-------+--------+--------------+-------------+-------------------+--------------------------+--------------------+------------+--------------------------+-------------------+
|T001          |C001       |P01       |PR01       |2026-06-01      |1200.50|THB     |success       |CRM          |2026-06-01 08:00:00|transactions_batch_001.csv|batch_2026_06_14_001|run_714826e2|2026-06-20 11:02:44.0

### Example 3: Data quality checks and quarantine split

DQ rules ใน lab นี้:

1. required `transaction_id`
2. required `customer_id`
3. valid `transaction_date`
4. positive `amount`
5. customer reference exists
6. product reference exists
7. property reference exists

Records ที่ไม่ผ่าน rule จะถูกเขียนลง quarantine table พร้อม `dq_issue_array`


In [8]:
# Add marker columns to check referential integrity with reference tables.
df_customer_ref = spark.read.table(customer_table).select("customer_id").withColumn("customer_ref_exists", F.lit(True))
df_product_ref = spark.read.table(product_table).select("product_id").withColumn("product_ref_exists", F.lit(True))
df_property_ref = spark.read.table(property_table).select("property_id").withColumn("property_ref_exists", F.lit(True))

df_ref_checked = (
    df_latest
    .join(df_customer_ref, on="customer_id", how="left")
    .join(df_product_ref, on="product_id", how="left")
    .join(df_property_ref, on="property_id", how="left")
)

df_dq_checked = (
    df_ref_checked
    .withColumn(
        "dq_issue_array_raw",
        F.array(
            F.when(F.col("transaction_id").isNull() | (F.length(F.col("transaction_id")) == 0), F.lit("MISSING_TRANSACTION_ID")),
            F.when(F.col("customer_id").isNull() | (F.length(F.col("customer_id")) == 0), F.lit("MISSING_CUSTOMER_ID")),
            F.when(F.col("transaction_date").isNull(), F.lit("INVALID_TRANSACTION_DATE")),
            F.when(F.col("amount").isNull() | (F.col("amount") <= 0), F.lit("INVALID_AMOUNT")),
            F.when(F.col("customer_id").isNotNull() & F.col("customer_ref_exists").isNull(), F.lit("CUSTOMER_NOT_FOUND")),
            F.when(F.col("product_id").isNotNull() & F.col("product_ref_exists").isNull(), F.lit("PRODUCT_NOT_FOUND")),
            F.when(F.col("property_id").isNotNull() & F.col("property_ref_exists").isNull(), F.lit("PROPERTY_NOT_FOUND")),
        )
    )
    .withColumn("dq_issue_array", F.expr("filter(dq_issue_array_raw, x -> x is not null)"))
    .drop("dq_issue_array_raw")
)

df_valid_latest = (
    df_dq_checked
    .filter(F.size(F.col("dq_issue_array")) == 0)
    .drop("customer_ref_exists", "product_ref_exists", "property_ref_exists")
)

df_invalid_latest = (
    df_dq_checked
    .filter(F.size(F.col("dq_issue_array")) > 0)
    .drop("customer_ref_exists", "product_ref_exists", "property_ref_exists")
    .withColumn("record_status", F.lit("invalid"))
)

df_quarantine = (
    df_invalid_latest
    .unionByName(df_duplicate_superseded, allowMissingColumns=True)
    .withColumn("quarantine_timestamp", F.current_timestamp())
)

# Tips:
# unionByName(..., allowMissingColumns=True): Combine DataFrames by column names; missing columns from either side will be filled with null

print("Valid latest rows:", df_valid_latest.count())
print("Quarantine rows:", df_quarantine.count())

df_quarantine.select("transaction_id", "customer_id", "product_id", "property_id", "amount", "transaction_date", "dq_issue_array", "record_status").orderBy("transaction_id").show(truncate=False)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 10, Finished, Available, Finished, False)

Valid latest rows: 4
Quarantine rows: 7
+--------------+-----------+----------+-----------+-------+----------------+-----------------------------------+-------------+
|transaction_id|customer_id|product_id|property_id|amount |transaction_date|dq_issue_array                     |record_status|
+--------------+-----------+----------+-----------+-------+----------------+-----------------------------------+-------------+
|T002          |C002       |P02       |PR01       |-50.00 |2026-06-01      |[INVALID_AMOUNT]                   |invalid      |
|T004          |C999       |P01       |PR01       |450.00 |2026-06-02      |[CUSTOMER_NOT_FOUND]               |invalid      |
|T005          |C001       |P99       |PR02       |300.00 |2026-06-03      |[PRODUCT_NOT_FOUND]                |invalid      |
|T006          |C004       |P02       |PR99       |1500.00|2026-06-04      |[PROPERTY_NOT_FOUND]               |invalid      |
|T007          |NULL       |P01       |PR01       |200.00 |2026-06-04  

### Example 4: Write Silver stage, quarantine table, and DQ summary

หลังจากแยก valid / invalid แล้ว เราจะเขียน output สำคัญ 3 ชุด:

- `day30_silver_transactions_stage` สำหรับ valid records ที่ผ่าน DQ
- `day30_quarantine_transactions` สำหรับ invalid/quarantine records
- `day30_dq_summary` สำหรับสรุปจำนวน records ต่อ issue reason


In [9]:
target_columns = [
    "transaction_id",
    "customer_id",
    "product_id",
    "property_id",
    "transaction_date",
    "amount",
    "currency",
    "payment_status",
    "source_system",
    "updated_at_ts",
    "batch_id",
    "run_id",
    "ingestion_timestamp",
    "source_file",
    "record_hash",
    "silver_updated_timestamp",
]

hash_columns = [
    "transaction_id",
    "customer_id",
    "product_id",
    "property_id",
    "transaction_date",
    "amount",
    "currency",
    "payment_status",
    "source_system",
]

df_valid_silver = (
    df_valid_latest
    # Cast hash columns to string, unpack them into concat_ws(), then create a SHA-256 record hash
    .withColumn("record_hash", F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in hash_columns]), 256))
    .withColumn("silver_updated_timestamp", F.current_timestamp())
    .select(*target_columns)
)

(
    df_valid_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_stage_table)
)

(
    df_quarantine
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(quarantine_table)
)

df_dq_summary = (
    df_quarantine
    .select(F.explode("dq_issue_array").alias("issue_reason"))
    .groupBy("issue_reason")
    .agg(F.count("*").alias("record_count"))
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("run_id", F.lit(run_id))
    .withColumn("summary_timestamp", F.current_timestamp())
    .select("run_id", "batch_id", "issue_reason", "record_count", "summary_timestamp")
)

(
    df_dq_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dq_summary_table)
)

spark.read.table(silver_stage_table).orderBy("transaction_id").show(truncate=False)
spark.read.table(dq_summary_table).orderBy("issue_reason").show(truncate=False)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 11, Finished, Available, Finished, False)

+--------------+-----------+----------+-----------+----------------+-------+--------+--------------+-------------+-------------------+--------------------+------------+--------------------------+--------------------------+----------------------------------------------------------------+--------------------------+
|transaction_id|customer_id|product_id|property_id|transaction_date|amount |currency|payment_status|source_system|updated_at_ts      |batch_id            |run_id      |ingestion_timestamp       |source_file               |record_hash                                                     |silver_updated_timestamp  |
+--------------+-----------+----------+-----------+----------------+-------+--------+--------------+-------------+-------------------+--------------------+------------+--------------------------+--------------------------+----------------------------------------------------------------+--------------------------+
|T001          |C001       |P01       |PR01       |2026

### Example 5: Incremental watermark window

ใน production เรามักจะไม่ process ข้อมูลทั้งหมดซ้ำทุกครั้ง แต่ใช้ watermark เพื่อเลือกเฉพาะ records ที่เปลี่ยนแปลงหลังรอบที่สำเร็จล่าสุด

Lab นี้ตั้ง initial watermark เป็น `2026-06-01 00:00:00` และใช้ max `updated_at_ts` ของ valid records เป็น upper bound


In [10]:
initial_watermark = datetime(2026, 6, 1, 0, 0, 0)

watermark_schema = T.StructType([
    T.StructField("pipeline_name", T.StringType(), False),
    T.StructField("last_successful_watermark", T.TimestampType(), False),
    T.StructField("last_successful_run_id", T.StringType(), True),
    T.StructField("updated_at", T.TimestampType(), True),
])

df_initial_watermark = spark.createDataFrame(
    [(pipeline_name, initial_watermark, None, datetime.now())],
    watermark_schema,
)

(
    df_initial_watermark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(watermark_table)
)

watermark_row = (
    spark.read.table(watermark_table)
    .filter(F.col("pipeline_name") == pipeline_name)
    .select("last_successful_watermark")
    .first()  # first() returns the first metadata row as a Python Row object.
)

last_successful_watermark = watermark_row["last_successful_watermark"]
run_upper_bound = df_valid_silver.agg(F.max("updated_at_ts").alias("run_upper_bound")).first()["run_upper_bound"]

if run_upper_bound is None:
    run_upper_bound = last_successful_watermark

print("Last successful watermark:", last_successful_watermark)
print("Run upper bound:", run_upper_bound)

df_valid_incremental = df_valid_silver.filter(
    (F.col("updated_at_ts") > F.lit(last_successful_watermark))
    & (F.col("updated_at_ts") <= F.lit(run_upper_bound))
)

print("Incremental valid rows:", df_valid_incremental.count())
df_valid_incremental.orderBy("transaction_id").show(truncate=False)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 12, Finished, Available, Finished, False)

Last successful watermark: 2026-06-01 00:00:00
Run upper bound: 2026-06-06 09:00:00
Incremental valid rows: 4
+--------------+-----------+----------+-----------+----------------+-------+--------+--------------+-------------+-------------------+--------------------+------------+--------------------------+--------------------------+----------------------------------------------------------------+--------------------------+
|transaction_id|customer_id|product_id|property_id|transaction_date|amount |currency|payment_status|source_system|updated_at_ts      |batch_id            |run_id      |ingestion_timestamp       |source_file               |record_hash                                                     |silver_updated_timestamp  |
+--------------+-----------+----------+-----------+----------------+-------+--------+--------------+-------------+-------------------+--------------------+------------+--------------------------+--------------------------+--------------------------------------

### Example 6: MERGE incremental records into Silver current table

เราจะสร้าง current target table ที่มีข้อมูลเดิม 2 records:

- `T001` เป็น record เก่าที่ควรถูก update เมื่อเจอข้อมูลใหม่จาก incoming source batch รอบนี้
- `T011` เป็น record เดิมที่ไม่ได้อยู่ใน incoming source batch รอบนี้ จึงควรถูกเก็บไว้ใน target table เหมือนเดิม

จากนั้นใช้ `MERGE` เพื่อ upsert valid incremental records เข้า Silver current target table โดยใช้ `transaction_id` เป็น business key

In [11]:
target_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("product_id", T.StringType(), True),
    T.StructField("property_id", T.StringType(), True),
    T.StructField("transaction_date", T.DateType(), True),
    T.StructField("amount", T.DecimalType(12, 2), True),
    T.StructField("currency", T.StringType(), True),
    T.StructField("payment_status", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("updated_at_ts", T.TimestampType(), True),
    T.StructField("batch_id", T.StringType(), True),
    T.StructField("run_id", T.StringType(), True),
    T.StructField("ingestion_timestamp", T.TimestampType(), True),
    T.StructField("source_file", T.StringType(), True),
    T.StructField("record_hash", T.StringType(), True),
    T.StructField("silver_updated_timestamp", T.TimestampType(), True),
])

existing_target_data = [
    (
        "T001", "C001", "P01", "PR01", date(2026, 5, 31), Decimal("999.00"), "THB", "success", "CRM",
        datetime(2026, 5, 31, 8, 0, 0), "initial_batch", "initial_run", datetime(2026, 5, 31, 8, 5, 0),
        "initial_load.csv", "old_hash_t001", datetime(2026, 5, 31, 8, 10, 0),
    ),
    (
        "T011", "C002", "P02", "PR01", date(2026, 5, 31), Decimal("700.00"), "THB", "success", "POS",
        datetime(2026, 5, 31, 9, 0, 0), "initial_batch", "initial_run", datetime(2026, 5, 31, 9, 5, 0),
        "initial_load.csv", "old_hash_t011", datetime(2026, 5, 31, 9, 10, 0),
    ),
]

df_existing_target = spark.createDataFrame(existing_target_data, target_schema)

(
    df_existing_target
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_current_table)
)

print("Target before MERGE:")
spark.read.table(silver_current_table).orderBy("transaction_id").show(truncate=False)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 13, Finished, Available, Finished, False)

Target before MERGE:
+--------------+-----------+----------+-----------+----------------+------+--------+--------------+-------------+-------------------+-------------+-----------+-------------------+----------------+-------------+------------------------+
|transaction_id|customer_id|product_id|property_id|transaction_date|amount|currency|payment_status|source_system|updated_at_ts      |batch_id     |run_id     |ingestion_timestamp|source_file     |record_hash  |silver_updated_timestamp|
+--------------+-----------+----------+-----------+----------------+------+--------+--------------+-------------+-------------------+-------------+-----------+-------------------+----------------+-------------+------------------------+
|T001          |C001       |P01       |PR01       |2026-05-31      |999.00|THB     |success       |CRM          |2026-05-31 08:00:00|initial_batch|initial_run|2026-05-31 08:05:00|initial_load.csv|old_hash_t001|2026-05-31 08:10:00     |
|T011          |C002       |P02    

In [12]:
# Register the incremental DataFrame as a temporary view so it can be used in Spark SQL MERGE.
# MERGE INTO uses SQL syntax, so the source DataFrame must be exposed as a table/view name first.
df_valid_incremental.select(*target_columns).createOrReplaceTempView("day30_valid_incremental_updates")

merge_sql = f"""
MERGE INTO {silver_current_table} AS tgt
USING day30_valid_incremental_updates AS src
ON tgt.transaction_id = src.transaction_id
WHEN MATCHED AND src.updated_at_ts > tgt.updated_at_ts THEN UPDATE SET
    tgt.customer_id = src.customer_id,
    tgt.product_id = src.product_id,
    tgt.property_id = src.property_id,
    tgt.transaction_date = src.transaction_date,
    tgt.amount = src.amount,
    tgt.currency = src.currency,
    tgt.payment_status = src.payment_status,
    tgt.source_system = src.source_system,
    tgt.updated_at_ts = src.updated_at_ts,
    tgt.batch_id = src.batch_id,
    tgt.run_id = src.run_id,
    tgt.ingestion_timestamp = src.ingestion_timestamp,
    tgt.source_file = src.source_file,
    tgt.record_hash = src.record_hash,
    tgt.silver_updated_timestamp = src.silver_updated_timestamp
WHEN NOT MATCHED THEN INSERT (
    transaction_id,
    customer_id,
    product_id,
    property_id,
    transaction_date,
    amount,
    currency,
    payment_status,
    source_system,
    updated_at_ts,
    batch_id,
    run_id,
    ingestion_timestamp,
    source_file,
    record_hash,
    silver_updated_timestamp
) VALUES (
    src.transaction_id,
    src.customer_id,
    src.product_id,
    src.property_id,
    src.transaction_date,
    src.amount,
    src.currency,
    src.payment_status,
    src.source_system,
    src.updated_at_ts,
    src.batch_id,
    src.run_id,
    src.ingestion_timestamp,
    src.source_file,
    src.record_hash,
    src.silver_updated_timestamp
)
"""

spark.sql(merge_sql)

print("Target after MERGE:")
df_silver_current = spark.read.table(silver_current_table)
df_silver_current.orderBy("transaction_id").show(truncate=False)
print("Silver current rows:", df_silver_current.count())

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 14, Finished, Available, Finished, False)

Target after MERGE:
+--------------+-----------+----------+-----------+----------------+-------+--------+--------------+-------------+-------------------+--------------------+------------+--------------------------+--------------------------+----------------------------------------------------------------+--------------------------+
|transaction_id|customer_id|product_id|property_id|transaction_date|amount |currency|payment_status|source_system|updated_at_ts      |batch_id            |run_id      |ingestion_timestamp       |source_file               |record_hash                                                     |silver_updated_timestamp  |
+--------------+-----------+----------+-----------+----------------+-------+--------+--------------+-------------+-------------------+--------------------+------------+--------------------------+--------------------------+----------------------------------------------------------------+--------------------------+
|T001          |C001       |P01    

### Example 7: Update watermark after successful MERGE

ใน production ต้อง update watermark หลัง target write/merge สำเร็จแล้วเท่านั้น

Lab นี้ update control table ด้วย `run_upper_bound` ที่เพิ่ง process สำเร็จ


In [13]:
df_new_watermark = spark.createDataFrame(
    [(pipeline_name, run_upper_bound, run_id, datetime.now())],
    watermark_schema,
)

(
    df_new_watermark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(watermark_table)
)

spark.read.table(watermark_table).show(truncate=False)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 15, Finished, Available, Finished, False)

+------------------------------+-------------------------+----------------------+--------------------------+
|pipeline_name                 |last_successful_watermark|last_successful_run_id|updated_at                |
+------------------------------+-------------------------+----------------------+--------------------------+
|day30_final_lakehouse_pipeline|2026-06-06 09:00:00      |run_714826e2          |2026-06-20 11:47:25.203298|
+------------------------------+-------------------------+----------------------+--------------------------+



### Example 8: Build Gold aggregate table

Gold table ใน lab นี้สรุป daily revenue ตาม:

- `transaction_date`
- `customer_region`
- `product_category`
- `source_system`

ก่อน write Gold เราจะใช้ `.explain()` เพื่อดู plan ของ join + aggregation แบบเบื้องต้น


In [14]:
df_gold_source = (
    spark.read.table(silver_current_table).alias("tx")
    .join(spark.read.table(customer_table).alias("c"), on="customer_id", how="left")
    .join(spark.read.table(product_table).alias("p"), on="product_id", how="left")
    .join(spark.read.table(property_table).alias("pr"), on="property_id", how="left")
)

df_gold_daily_revenue = (
    df_gold_source
    .groupBy("transaction_date", "customer_region", "product_category", "source_system")
    .agg(
        F.count("transaction_id").alias("transaction_count"),
        F.countDistinct("customer_id").alias("unique_customer_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
    )
    .withColumn("gold_updated_timestamp", F.current_timestamp())
)

# Explain before writing to understand join + aggregation behavior.
df_gold_daily_revenue.explain(mode="formatted")

(
    df_gold_daily_revenue
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

spark.read.table(gold_table).orderBy("transaction_date", "customer_region", "product_category").show(truncate=False)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 16, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (20)
+- HashAggregate (19)
   +- Exchange (18)
      +- HashAggregate (17)
         +- HashAggregate (16)
            +- Exchange (15)
               +- HashAggregate (14)
                  +- Project (13)
                     +- BroadcastHashJoin LeftOuter BuildRight (12)
                        :- Project (9)
                        :  +- BroadcastHashJoin LeftOuter BuildRight (8)
                        :     :- Project (5)
                        :     :  +- BroadcastHashJoin LeftOuter BuildRight (4)
                        :     :     :- Scan parquet spark_catalog.lh_pyspark_challenge.day30_silver_transactions_current (1)
                        :     :     +- BroadcastExchange (3)
                        :     :        +- Scan parquet spark_catalog.lh_pyspark_challenge.day30_dim_properties (2)
                        :     +- BroadcastExchange (7)
                        :        +- Scan parquet spark_catalog.lh_pyspark_challenge.day30_dim_pr

### Example 9: Schema drift detection without breaking Run all

ในตัวอย่างนี้เราจะไม่ append ข้อมูลที่ schema ไม่ตรงเข้า table จริง แต่จะ simulate incoming schema ที่มี:

- missing column: `currency`
- extra column: `sales_channel`

แล้วเขียนผลลง schema validation log


In [15]:
expected_raw_columns = [
    "transaction_id",
    "customer_id",
    "product_id",
    "property_id",
    "transaction_date",
    "amount",
    "currency",
    "payment_status",
    "source_system",
    "updated_at",
    "source_file",
]

incoming_drift_columns = [
    "transaction_id",
    "customer_id",
    "product_id",
    "property_id",
    "transaction_date",
    "amount",
    "payment_status",
    "source_system",
    "updated_at",
    "source_file",
    "sales_channel",
]

# set(...): Convert column lists to sets so we can compare missing/extra column names.
# set_a - set_b: Find columns that exist in the first set but not in the second set.
# list(...): Convert the set result back to a list for logging and display.
# sorted(...): Sort the list to make the output stable and readable.
missing_columns = sorted(list(set(expected_raw_columns) - set(incoming_drift_columns)))
extra_columns = sorted(list(set(incoming_drift_columns) - set(expected_raw_columns)))
schema_status = "PASS" if not missing_columns and not extra_columns else "REVIEW_REQUIRED"

schema_log_schema = T.StructType([
    T.StructField("run_id", T.StringType(), False),
    T.StructField("batch_id", T.StringType(), False),
    T.StructField("table_name", T.StringType(), False),
    T.StructField("schema_status", T.StringType(), False),
    T.StructField("missing_columns", T.ArrayType(T.StringType()), True),
    T.StructField("extra_columns", T.ArrayType(T.StringType()), True),
    T.StructField("checked_at", T.TimestampType(), True),
])

df_schema_log = spark.createDataFrame(
    [(run_id, batch_id, bronze_table, schema_status, missing_columns, extra_columns, datetime.now())],
    schema_log_schema,
)

(
    df_schema_log
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(schema_log_table)
)

spark.read.table(schema_log_table).show(truncate=False)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 17, Finished, Available, Finished, False)

+------------+--------------------+-------------------------+---------------+---------------+---------------+--------------------------+
|run_id      |batch_id            |table_name               |schema_status  |missing_columns|extra_columns  |checked_at                |
+------------+--------------------+-------------------------+---------------+---------------+---------------+--------------------------+
|run_714826e2|batch_2026_06_14_001|day30_bronze_transactions|REVIEW_REQUIRED|[currency]     |[sales_channel]|2026-06-20 11:57:07.334786|
+------------+--------------------+-------------------------+---------------+---------------+---------------+--------------------------+



### Example 10: Write audit run log

Audit log ควรบอกภาพรวมของ pipeline run เช่น:

- Bronze rows
- valid rows
- quarantine rows
- Silver current rows
- Gold rows
- DQ issue count
- watermark window
- status


In [16]:
bronze_rows = spark.read.table(bronze_table).count()
valid_rows = spark.read.table(silver_stage_table).count()
quarantine_rows = spark.read.table(quarantine_table).count()
silver_current_rows = spark.read.table(silver_current_table).count()
gold_rows = spark.read.table(gold_table).count()
dq_issue_rows = spark.read.table(dq_summary_table).agg(F.sum("record_count").alias("total_issues")).first()["total_issues"]

audit_schema = T.StructType([
    T.StructField("run_id", T.StringType(), False),
    T.StructField("batch_id", T.StringType(), False),
    T.StructField("pipeline_name", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("lower_watermark", T.TimestampType(), True),
    T.StructField("upper_watermark", T.TimestampType(), True),
    T.StructField("bronze_rows", T.LongType(), True),
    T.StructField("valid_rows", T.LongType(), True),
    T.StructField("quarantine_rows", T.LongType(), True),
    T.StructField("silver_current_rows", T.LongType(), True),
    T.StructField("gold_rows", T.LongType(), True),
    T.StructField("dq_issue_rows", T.LongType(), True),
    T.StructField("run_timestamp", T.TimestampType(), True),
])

df_audit_log = spark.createDataFrame(
    [(
        run_id,
        batch_id,
        pipeline_name,
        "SUCCESS",
        last_successful_watermark,
        run_upper_bound,
        bronze_rows,
        valid_rows,
        quarantine_rows,
        silver_current_rows,
        gold_rows,
        int(dq_issue_rows or 0),  # Use 0 if dq_issue_rows is None or empty
        datetime.now(),
    )],
    audit_schema,
)

(
    df_audit_log
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(audit_log_table)
)

spark.read.table(audit_log_table).show(truncate=False)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 18, Finished, Available, Finished, False)

+------------+--------------------+------------------------------+-------+-------------------+-------------------+-----------+----------+---------------+-------------------+---------+-------------+--------------------------+
|run_id      |batch_id            |pipeline_name                 |status |lower_watermark    |upper_watermark    |bronze_rows|valid_rows|quarantine_rows|silver_current_rows|gold_rows|dq_issue_rows|run_timestamp             |
+------------+--------------------+------------------------------+-------+-------------------+-------------------+-----------+----------+---------------+-------------------+---------+-------------+--------------------------+
|run_714826e2|batch_2026_06_14_001|day30_final_lakehouse_pipeline|SUCCESS|2026-06-01 00:00:00|2026-06-06 09:00:00|11         |4         |7              |5                  |5        |7            |2026-06-20 12:01:23.077533|
+------------+--------------------+------------------------------+-------+-------------------+------

## Quick recap

| Question | Answer |
|---|---|
| Bronze table ใช้ทำอะไร? | เก็บข้อมูลจาก source ในรูปแบบ raw as ingested พร้อม audit columns เพื่อ trace และ reprocess ได้ |
| Silver stage ทำอะไร? | Clean, cast, standardize, deduplicate และเตรียม record สำหรับ DQ/MERGE |
| Quarantine table ต่างจาก drop records ยังไง? | Quarantine เก็บ invalid records พร้อม reason ส่วน drop ทำให้เกิด silent data loss |
| Watermark ควร update ตอนไหน? | หลัง target write / MERGE สำเร็จแล้วเท่านั้น |
| MERGE ช่วยเรื่องอะไร? | ทำ incremental upsert ให้ rerun-safe กว่าการ append โดยไม่ตรวจ business key |
| Gold table ควรเน้นอะไร? | ข้อมูลสรุปที่ business/analytics ใช้ต่อได้ โดยมาจาก Silver ที่ผ่าน cleansing และ DQ แล้ว |
| Schema drift ควรจัดการยังไง? | ตรวจและ log missing/extra/type changes ก่อนกระทบ target table หรือ downstream logic |
| `.explain()` ใช้ตรงไหนใน final pipeline? | ใช้ดู execution plan ของ join/aggregation หรือ step ที่มี performance risk |


## Exercises

### Exercise 1: Add high-value transaction summary

สร้าง Gold summary เพิ่มเติมเพื่อดูจำนวน high-value transactions

Requirements:

- อ่านจาก `day30_silver_transactions_current`
- สร้าง column `is_high_value` เมื่อ `amount >= 1000`
- group by `is_high_value`
- แสดง `transaction_count` และ `total_amount`

Expected result:

- เห็น summary แยก high-value vs normal transactions


In [17]:
df_high_value_summary = (
    spark.read.table(silver_current_table)
    .withColumn("is_high_value", F.col("amount") >= F.lit(1000))
    .groupBy("is_high_value")
    .agg(
        F.count("transaction_id").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
    )
    .orderBy(F.col("is_high_value").desc())
)

df_high_value_summary.show(truncate=False)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 19, Finished, Available, Finished, False)

+-------------+-----------------+------------+
|is_high_value|transaction_count|total_amount|
+-------------+-----------------+------------+
|true         |1                |1200.50     |
|false        |4                |3100.00     |
+-------------+-----------------+------------+



### Exercise 2: Validate MERGE rerun safety

ลอง MERGE incoming source batch ชุดเดิมซ้ำอีกครั้ง แล้วเช็คว่า row count ของ current table ไม่เพิ่มผิดปกติ

Requirements:

- เก็บ count ก่อน MERGE
- รัน `spark.sql(merge_sql)` อีกครั้ง
- เก็บ count หลัง MERGE
- count ก่อนและหลังควรเท่ากัน

Expected result:

- `before_count == after_count` ควรเป็น `True`


In [18]:
before_count = spark.read.table(silver_current_table).count()

spark.sql(merge_sql)

after_count = spark.read.table(silver_current_table).count()

print("Before MERGE rerun:", before_count)
print("After MERGE rerun:", after_count)
print("Rerun-safe row count:", before_count == after_count)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 20, Finished, Available, Finished, False)

Before MERGE rerun: 5
After MERGE rerun: 5
Rerun-safe row count: True


### Exercise 3: Create a batch-to-streaming extension checklist

สร้าง DataFrame เล็ก ๆ เพื่อสรุปว่า ถ้าจะต่อ Bronze batch ingestion ให้เป็น Structured Streaming ต้องคิดเรื่องอะไรบ้าง

Requirements:

- สร้าง columns: `area`, `design_decision`, `monitoring_note`
- อย่างน้อย 5 rows
- แสดงผลด้วย `.show(truncate=False)`

Expected result:

- ได้ checklist ที่อธิบาย source, sink, checkpoint, schema, trigger และ monitoring แบบสั้น ๆ


In [19]:
streaming_extension_rows = [
    ("source", "Use readStream with explicit schema from landing/event files", "Monitor input rows per micro-batch"),
    ("sink", "Write append-only events to Bronze Delta table", "Monitor output rows and failed writes"),
    ("checkpoint", "Use stable checkpoint path under Lakehouse Files", "Do not delete checkpoint unless intentionally replaying"),
    ("schema", "Fail or quarantine unexpected schema changes", "Log missing/extra columns before downstream processing"),
    ("trigger", "Choose trigger interval based on latency and small-file trade-off", "Too frequent triggers can create many small files"),
    ("operations", "Track query status, batch duration, lag, and errors", "Alert when batches fail or processing time keeps increasing"),
]

df_streaming_extension_checklist = spark.createDataFrame(
    streaming_extension_rows,
    ["area", "design_decision", "monitoring_note"],
)

df_streaming_extension_checklist.show(truncate=False)

StatementMeta(, 326df9e2-cf7c-4b64-ab54-15604e2e21d9, 21, Finished, Available, Finished, False)

+----------+-----------------------------------------------------------------+-----------------------------------------------------------+
|area      |design_decision                                                  |monitoring_note                                            |
+----------+-----------------------------------------------------------------+-----------------------------------------------------------+
|source    |Use readStream with explicit schema from landing/event files     |Monitor input rows per micro-batch                         |
|sink      |Write append-only events to Bronze Delta table                   |Monitor output rows and failed writes                      |
|checkpoint|Use stable checkpoint path under Lakehouse Files                 |Do not delete checkpoint unless intentionally replaying    |
|schema    |Fail or quarantine unexpected schema changes                     |Log missing/extra columns before downstream processing     |
|trigger   |Choose trigger 

## Common mistakes

1. **เขียน pipeline เป็น cell ใหญ่ cell เดียว**

   ทำให้ debug ยากและหาไม่ได้ว่า fail ที่ Bronze, Silver, DQ, MERGE หรือ Gold ควรแบ่ง step ให้ตรวจ row count และ preview ได้เป็นช่วง ๆ

2. **Filter invalid records ทิ้งโดยไม่เขียน quarantine**

   การทิ้ง bad records เงียบ ๆ ทำให้ audit ย้อนหลังไม่ได้ และอาจทำให้ business เชื่อว่าข้อมูลครบ ทั้งที่จริง ๆ หายไปบางส่วน

3. **Update watermark ก่อน target write สำเร็จ**

   ถ้า watermark ถูกเลื่อนไปข้างหน้าแล้ว แต่ `MERGE` หรือ target write fail ข้อมูลใน window นั้นอาจถูก skip เพราะรอบถัดไปจะเริ่มอ่านจาก watermark ใหม่

4. **ใช้ append กับ incremental target โดยไม่ตรวจ business key**

   ถ้า incoming source batch ส่ง update ของ transaction เดิมมา append จะสร้าง duplicate current records ทันที ควรใช้ MERGE/UPSERT เมื่อ target ต้องเป็น latest state

5. **คิดว่า Bronze ต้อง clean ทุกอย่างตั้งแต่ต้น**

   Bronze ควรเน้น traceability และ raw preservation มากกว่า business cleansing ลึก ๆ ส่วน cleansing/DQ ควรอยู่ใน Silver

6. **มอง schema drift เป็นแค่ technical error**

   Schema drift ไม่ใช่แค่ technical error แต่เป็น data contract issue ระหว่าง source กับ pipeline ต้อง log, review และตัดสินใจว่าจะ reject, quarantine, ปรับ schema ให้ตรงกับ target หรือ evolve schema ตาม source หลังจาก review แล้วว่าเป็น change ที่ยอมรับได้

7. **ดูผลลัพธ์อย่างเดียวโดยไม่ดู execution plan**

   Join และ aggregation อาจให้ผลถูกแต่ช้ามาก ควรใช้ `.explain()` ร่วมกับ row counts, key distribution, partition distribution และสัญญาณของ data skew เมื่อเริ่ม debug performance


## Expected Output / Success Criteria

เมื่อจบ Day 30 ควรทำได้:

- อธิบาย end-to-end flow จาก mock source → Bronze → Silver → quarantine → current target → Gold ได้
- สร้าง Bronze table พร้อม audit columns ได้
- Clean และ cast data types ใน Silver ได้
- ใช้ window function เพื่อเลือก latest record ต่อ business key ได้
- Apply DQ rules และเขียน invalid records ลง quarantine table พร้อม `dq_issue_array` ได้
- ใช้ watermark filter ด้วย pattern `lower < updated_at_ts <= upper` ได้
- ใช้ `MERGE INTO` เพื่อ upsert incremental valid records เข้า target table ได้
- สร้าง Gold aggregate table จาก Silver current table ได้
- เขียน audit run log และ schema validation log ได้
- ใช้ `.explain(mode="formatted")` เพื่อตรวจ execution plan ของ join/aggregation step ที่มี performance risk ได้
- อธิบายได้ว่า batch Bronze ingestion จะต่อยอดเป็น Structured Streaming ingestion ได้อย่างไร

## Final takeaway

Final Lakehouse Pipeline คือการเอา concept หลายวันมาต่อกันเป็น pipeline ที่ตรวจสอบได้ ไม่ใช่แค่ชุดคำสั่งที่รันแล้วได้ผลลัพธ์ที่ถูกต้องเพียงครั้งเดียว

> Pipeline ที่ไว้ใจได้ต้องมี traceability, data quality, quarantine, incremental safety, audit log และแนวทางการ debug ที่ชัดเจน

สิ่งที่ควรจำหลังจบ challenge:

- Bronze เก็บหลักฐานของข้อมูลและช่วย trace กลับไปหา source/run ได้
- Silver ทำให้ข้อมูล clean, standardized และพร้อมสำหรับ validation/MERGE
- Quarantine ทำให้ bad records ไม่ถูก drop ทิ้งแบบไม่มี trace
- Watermark + MERGE ช่วยให้ incremental pipeline rerun-safe
- Gold ควรอ่านจากข้อมูลที่ผ่าน cleansing, DQ และ transformation logic แล้ว ไม่ใช่ raw source ตรง ๆ
- `.explain()` ช่วยดู execution plan ส่วน audit log ช่วยตรวจ run status, row counts และจุดที่อาจเกิดปัญหา
- Streaming ไม่ได้มาแทน batch ในทุกกรณี แต่สามารถต่อยอด Bronze ingestion ได้เมื่อ source pattern และ operational readiness พร้อม


## Optional cleanup

In [ ]:
# tables_to_drop = [
#     bronze_table,
#     silver_stage_table,
#     silver_current_table,
#     quarantine_table,
#     gold_table,
#     dq_summary_table,
#     audit_log_table,
#     watermark_table,
#     schema_log_table,
#     customer_table,
#     product_table,
#     property_table,
# ]
#
# for table_name in tables_to_drop:
#     spark.sql(f"DROP TABLE IF EXISTS {table_name}")
#     print(f"Dropped table: {table_name}")